In [ ]:
import numpy as np
import pandas as pd

In [ ]:
spectra = pd.read_csv(f'/home/s1856122/XGBoost/ready_to_use_484_kinase_karaman_specific_sim50/train_ready2use_merged_4datasets_deduplicated_no_karaman_sim0.5.csv')
onehot_path = f"../../dataset/karaman/Karaman_sim0.5_morgan_one_hot_from_merge.csv"
decoder_path = f"../../dataset/karaman/Karaman_sim0.5_sae_of1024d_decoder_all_from_merge.pt"
embedding_output = f"../../dataset/karaman/Karaman_sim0.5_SAE_latent_top95_features_from_merge.csv"

In [ ]:
def parse_one_hot(column: pd.Series) -> np.ndarray:
    return np.array([list(map(int, list(x))) for x in column], dtype=np.int8)
    
spectra_morgan_oh = pd.DataFrame(parse_one_hot(spectra['Morgan']))
spectra_morgan_oh.to_csv(onehot_path,header=None,index=False)

In [ ]:
# ===== Imports =====
from __future__ import annotations
import os, gc, json
from dataclasses import dataclass
from pathlib import Path
from typing import List, Tuple, Dict
import hashlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import pandas as pd
import numpy as np
from tqdm import tqdm

# -------------------------------------------------
# 1. Configuration
# -------------------------------------------------
@dataclass(frozen=True)
class SAEConfig:
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    input_dim: int       = 4096
    latent_dim: int      = 1024
    csv_path: str        =  onehot_path
    chunk_rows: int      = 50_000
    batch_size: int      = 1024
    model_path: str      = "./LSAE_1900k_chembl_model.pt"
    idx_path: str        = "./LSAE_chembl_1900k_topk_of1024d_top99.pt"

# -------------------------------------------------
# 2. SAE Definition
# -------------------------------------------------
class SparseLinearAE(nn.Module):
    def __init__(self, cfg: SAEConfig):
        super().__init__()
        self.encoder = nn.Linear(cfg.input_dim, cfg.latent_dim, bias=True)
        self.decoder = nn.Linear(cfg.latent_dim, cfg.input_dim, bias=True)

    def forward(self, x):
        z = self.encoder(x)
        x_hat = self.decoder(z)
        return x_hat, z

# -------------------------------------------------
# 3. Load Model & Latent Indices
# -------------------------------------------------
def load_model_and_idx(cfg: SAEConfig):
    print(f"[info] Using device: {cfg.device}")
    
    model = SparseLinearAE(cfg).to(cfg.device)
    checkpoint = torch.load(cfg.model_path, map_location=cfg.device, weights_only=True)
    model.load_state_dict(checkpoint)
    model.eval()

    keep_idx = torch.load(cfg.idx_path, map_location=cfg.device, weights_only=True).long()

    #----------------
    keep_idx = keep_idx[:466]  # Select the top-k latent features covering 95% of bit=1 entries (466 features)
    #------------------

    print(f"[info] loaded model + {len(keep_idx)} latent indices on {cfg.device}")
    return model, keep_idx

# -------------------------------------------------
# 4. Deduplication and Mapping Functions
# -------------------------------------------------
def deduplicate_csv(csv_path: str, chunk_size: int = 50000) -> Tuple[pd.DataFrame, np.ndarray]:
    """
    Deduplicate the CSV and return unique rows and their mapping.
    
    Returns:
        unique_df: Deduplicated DataFrame.
        mapping: Array mapping original row indices to deduplicated row indices.
    """
    print("[info] Reading and deduplicating CSV...")
    
    # Read the CSV in chunks and remove duplicate rows
    unique_rows = {}  # hash -> (row_data, first_occurrence_idx)
    row_to_unique_idx = []  # Mapping from original row indices to deduplicated row indices
    unique_count = 0
    total_rows = 0
    
    # Count the total number of rows first
    with open(csv_path, 'rb') as f:
        total_lines = sum(1 for _ in f)
    
    # Process the CSV chunk by chunk
    for chunk in tqdm(pd.read_csv(csv_path, header=None, chunksize=chunk_size), 
                      desc="Deduplicating", total=(total_lines//chunk_size + 1)):
        
        for idx, row in chunk.iterrows():
            # Create a hash for the row
            row_hash = hashlib.md5(row.values.tobytes()).hexdigest()
            
            if row_hash not in unique_rows:
                # New unique row
                unique_rows[row_hash] = (row.values, unique_count)
                row_to_unique_idx.append(unique_count)
                unique_count += 1
            else:
                # Duplicate row; map it to the existing unique row
                _, existing_unique_idx = unique_rows[row_hash]
                row_to_unique_idx.append(existing_unique_idx)
            
            total_rows += 1
    
    # Build the deduplicated DataFrame
    unique_data = np.array([data for data, _ in unique_rows.values()])
    unique_df = pd.DataFrame(unique_data)
    
    print(f"[info] Original rows: {total_rows}, Unique rows: {unique_count}")
    print(f"[info] Deduplication ratio: {unique_count/total_rows:.3f}")
    
    return unique_df, np.array(row_to_unique_idx)

# -------------------------------------------------
# 5. Simplified Dataset for Deduplicated Data
# -------------------------------------------------
class UniqueFingerprintDataset(Dataset):
    def __init__(self, df: pd.DataFrame):
        self.data = torch.from_numpy(df.values.astype(np.uint8)).float()
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        return self.data[idx]

# -------------------------------------------------
# 6. Optimized Feature Extraction Function
# -------------------------------------------------
def extract_and_save_with_dedup(cfg: SAEConfig):
    # 1. Deduplicate the input data
    unique_df, mapping = deduplicate_csv(cfg.csv_path, cfg.chunk_rows)
    
    # 2. Load the model
    model, keep_idx = load_model_and_idx(cfg)
    
    # 3. Extract features from the deduplicated data
    unique_ds = UniqueFingerprintDataset(unique_df)
    loader = DataLoader(
        unique_ds, 
        batch_size=cfg.batch_size, 
        num_workers=0,
        pin_memory=True if cfg.device.type == 'cuda' else False
    )
    
    unique_feats: List[torch.Tensor] = []
    
    print(f"[info] Extracting features from {len(unique_ds)} unique molecules...")
    with torch.no_grad():
        for xb in tqdm(loader, desc=f"encoding unique molecules on {cfg.device}"):
            xb = xb.to(cfg.device, non_blocking=True)
            z = model.encoder(xb)[:, keep_idx]  # [B, k]
            unique_feats.append(z.cpu())
            
            # Periodically clear the GPU cache
            if len(unique_feats) % 20 == 0:
                torch.cuda.empty_cache() if cfg.device.type == 'cuda' else None
    
    # 4. Concatenate features for unique molecules
    unique_latent = torch.cat(unique_feats).numpy()  # [unique_count, k]
    print(f"[info] Unique features shape: {unique_latent.shape}")
    
    # 5. Restore the original row order using the mapping
    print("[info] Restoring original order...")
    original_latent = unique_latent[mapping]  # [original_count, k]
    print(f"[info] Restored features shape: {original_latent.shape}")
    
    # 6. Save the results
    pd.DataFrame(original_latent).to_csv(embedding_output, index=False)
    print(f"[save] latent features → {embedding_output}")

    dec_sub = model.decoder.weight[:, keep_idx].cpu()
    decoder_data = {
        "idx": keep_idx.cpu(), 
        "decoder_sub": dec_sub, 
        "bias": model.decoder.bias.cpu()
    }
    torch.save(decoder_data, decoder_path)
    print(f"[save] decoder submatrix → {decoder_path}")
    
    # Clear GPU memory
    if cfg.device.type == 'cuda':
        torch.cuda.empty_cache()
        print(f"[info] GPU memory cleared")

# -------------------------------------------------
# 7. main
# -------------------------------------------------
if __name__ == "__main__":
    cfg = SAEConfig()
    
    if cfg.device.type == 'cuda':
        print(f"[info] GPU: {torch.cuda.get_device_name()}")
        print(f"[info] GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    
    extract_and_save_with_dedup(cfg)

[info] GPU: NVIDIA GeForce RTX 5090
[info] GPU Memory: 33.7 GB
[info] Reading and deduplicating CSV...


Deduplicating: 100%|██████████| 14/14 [01:37<00:00,  6.95s/it]


[info] Original rows: 673463, Unique rows: 300172
[info] Deduplication ratio: 0.446
[info] Using device: cuda
[info] loaded model + 466 latent indices on cuda
[info] Extracting features from 300172 unique molecules...


encoding unique molecules on cuda: 100%|██████████| 294/294 [00:00<00:00, 343.77it/s]


[info] Unique features shape: (300172, 466)
[info] Restoring original order...
[info] Restored features shape: (673463, 466)
[save] latent features → ../../dataset/karaman/Karaman_sim0.5_SAE_latent_top95_features_from_merge.csv
[save] decoder submatrix → ../../dataset/karaman/Karaman_sim0.5_sae_of1024d_decoder_all_from_merge.pt
[info] GPU memory cleared
